In [9]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv

# .env 불러오기
load_dotenv("backend/.env")  # ipynb 위치에 맞춰 경로 조정 필요

project = os.getenv("GCP_PROJECT_ID")
location = os.getenv("GCP_LOCATION")          # us-central1 등
endpoint_id = os.getenv("GCP_VERTEX_ENDPOINT_ID")

# 인증 키 경로도 자동 적용
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

In [10]:
endpoint = aiplatform.Endpoint(
    endpoint_name=f"projects/{project}/locations/{location}/endpoints/{endpoint_id}"
)

In [12]:
import os
print("Project:", os.getenv("GCP_PROJECT_ID"))
print("Creds:", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))
print("Endpoint:", os.getenv("GCP_VERTEX_ENDPOINT_ID"))

Project: seoulpoi
Creds: /Users/yeonsoo/Desktop/gcp 수정(백엔드)/backend/seoulpoi-efc1c9e6c116.json
Endpoint: 7675799525155405824


In [14]:
import os

IMG_DIR = "all"
for fn in os.listdir(IMG_DIR):
    fp = os.path.join(IMG_DIR, fn)
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / (1024*1024)
        print(f"{fn}: {size_mb:.2f} MB")

HF030003_0201_2736.JPG: 0.22 MB
다운로드 (19).jpg: 0.13 MB
HF030001_0101_0468.jpg: 0.63 MB
HF030003_0601_0202.JPG: 0.76 MB
다운로드 (1).jpg: 0.28 MB
HF030003_0201_1388.JPG: 0.13 MB
다운로드 (23).jpg: 0.08 MB
HF030004_0401_0592 - 복사본.JPG: 0.35 MB
HF030003_0201_1389.JPG: 0.13 MB
HF030003_0601_0203.JPG: 0.74 MB
Image (5).jpg: 4.35 MB
HF030001_0101_0469.jpg: 0.31 MB
kim-sung-jin-BKgeiTfxLdA-unsplash.jpg: 3.76 MB
HF030004_0401_0629 - 복사본.JPG: 0.34 MB
HF030001_0301_0031.JPG: 0.46 MB
HF030001_0301_0025.JPG: 0.35 MB
HF030004_0101_0276.jpg: 0.38 MB
HF030004_0201_0278.JPG: 0.20 MB
다운로드 (15).jpg: 0.12 MB
daniel-bernard-RbaGeW_LbGs-unsplash.jpg: 1.83 MB
HF030004_0301_0262 - 복사본.JPG: 0.36 MB
HF030004_0201_0279.JPG: 0.28 MB
HF030004_0301_0551 - 복사본.JPG: 0.23 MB
HF030003_0201_2734.JPG: 0.22 MB
pexels-lepei-17191919.jpg: 2.64 MB
HF030003_0201_2730.JPG: 0.18 MB
HF030001_0301_0020.JPG: 0.45 MB
HF030003_0601_0364.JPG: 0.89 MB
HF030001_0101_0308.jpg: 0.37 MB
HF030003 _0301_0004

In [17]:
import os
import base64
from PIL import Image
from google.cloud import aiplatform
from dotenv import load_dotenv
import pandas as pd

# 1. 환경 변수 불러오기
load_dotenv("backend/.env")
project = os.getenv("GCP_PROJECT_ID")
location = os.getenv("GCP_LOCATION")            # ex) us-central1
endpoint_id = os.getenv("GCP_VERTEX_ENDPOINT_ID")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# 2. 엔드포인트 객체 생성
endpoint = aiplatform.Endpoint(
    endpoint_name=f"projects/{project}/locations/{location}/endpoints/{endpoint_id}"
)

# 3. 이미지 리사이즈 함수 (RGBA → RGB 변환 포함)
def resize_image(input_path, output_path, max_size=(1024,1024), quality=85):
    img = Image.open(input_path)
    img.thumbnail(max_size)

    # RGBA → RGB 변환 (투명 배경 흰색으로 채움)
    if img.mode == "RGBA":
        background = Image.new("RGB", img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3])  # A 채널 마스크
        img = background
    else:
        img = img.convert("RGB")

    img.save(output_path, format="JPEG", quality=quality)

# 4. 입력/출력 폴더 준비
IMG_DIR = "all"             # 원본 이미지 폴더
OUT_DIR = "all_resized"     # 리사이즈된 이미지 폴더
os.makedirs(OUT_DIR, exist_ok=True)

# 5. 리사이즈 + 예측 요청
results = []
for fn in os.listdir(IMG_DIR):
    if fn.lower().endswith((".jpg", ".jpeg", ".png")):
        inp = os.path.join(IMG_DIR, fn)
        out = os.path.join(OUT_DIR, fn)

        # 리사이즈 저장
        resize_image(inp, out)
        print(f"Resized {fn} -> {out}")

        # 예측 요청
        with open(out, "rb") as f:
            content = f.read()
            encoded_content = base64.b64encode(content).decode("utf-8")

        instance = {"content": encoded_content}
        response = endpoint.predict(instances=[instance])

        predictions = response.predictions
        if predictions:
            label = predictions[0]["displayNames"][0]
            confidence = predictions[0]["confidences"][0]
            results.append((fn, label, confidence))

# 6. 결과 DataFrame + CSV 저장
df = pd.DataFrame(results, columns=["filename", "label", "confidence"])
print(df)
df.to_csv("batch_results.csv", index=False, encoding="utf-8-sig")
print("✅ batch_results.csv 저장 완료")

Resized HF030003_0201_2736.JPG -> all_resized/HF030003_0201_2736.JPG
Resized 다운로드 (19).jpg -> all_resized/다운로드 (19).jpg
Resized HF030001_0101_0468.jpg -> all_resized/HF030001_0101_0468.jpg
Resized HF030003_0601_0202.JPG -> all_resized/HF030003_0601_0202.JPG
Resized 다운로드 (1).jpg -> all_resized/다운로드 (1).jpg
Resized HF030003_0201_1388.JPG -> all_resized/HF030003_0201_1388.JPG
Resized 다운로드 (23).jpg -> all_resized/다운로드 (23).jpg
Resized HF030004_0401_0592 - 복사본.JPG -> all_resized/HF030004_0401_0592 - 복사본.JPG
Resized HF030003_0201_1389.JPG -> all_resized/HF030003_0201_1389.JPG
Resized HF030003_0601_0203.JPG -> all_resized/HF030003_0601_0203.JPG
Resized Image (5).jpg -> all_resized/Image (5).jpg
Resized HF030001_0101_0469.jpg -> all_resized/HF030001_0101_0469.jpg
Resized kim-sung-jin-BKgeiTfxLdA-unsplash.jpg -> all_resized/kim-sung-jin-BKgeiTfxLdA-unsplash.jpg
Resized HF030004_0401_0629 - 복사본.JPG -> all_resized/HF030004_0401_0629 - 복사본.JPG
Resi

In [19]:
import os
import base64
from google.cloud import aiplatform
from dotenv import load_dotenv
import pandas as pd

# 1. 환경 변수 불러오기
load_dotenv("../backend/.env")
project = os.getenv("GCP_PROJECT_ID")
location = os.getenv("GCP_LOCATION")          # 예: us-central1
endpoint_id = os.getenv("GCP_VERTEX_ENDPOINT_ID")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# 2. 엔드포인트 객체 생성
endpoint = aiplatform.Endpoint(
    endpoint_name=f"projects/{project}/locations/{location}/endpoints/{endpoint_id}"
)

# 3. 이미지 폴더 테스트
IMG_DIR = "all_resized"
results = []

for filename in os.listdir(IMG_DIR):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        file_path = os.path.join(IMG_DIR, filename)
        with open(file_path, "rb") as f:
            content = f.read()
            encoded_content = base64.b64encode(content).decode("utf-8")

        # base64 인코딩된 문자열을 content에 넣음
        instance = {"content": encoded_content}
        response = endpoint.predict(instances=[instance])

        predictions = response.predictions
        if predictions:
            labels = predictions[0]["displayNames"]
            scores = predictions[0]["confidences"]

            # confidence가 가장 높은 라벨 선택
            max_idx = scores.index(max(scores))
            label = labels[max_idx]
            confidence = scores[max_idx]

            results.append((filename, label, confidence))

# 4. 결과 DataFrame
df = pd.DataFrame(results, columns=["filename", "label", "confidence"])
df

,filename,label,confidence
0,HF030003_0201_2736.JPG,NSeoulTower,0.984089
1,다운로드 (19).jpg,GyeongbokgungPalace,0.983404
2,HF030001_0101_0468.jpg,63Building,0.905346
3,HF030003_0601_0202.JPG,NSeoulTower,0.980123
4,다운로드 (1).jpg,GyeongbokgungPalace,0.992038
...,...,...,...
241,HF030004_0301_1724 - 복사본.JPG,LotteWorldTower,0.656011
242,HF030004_0301_1731 - 복사본.JPG,LotteWorldTower,0.773828
243,HF030001_0101_0471.jpg,63Building,0.977282
244,HF030004_0401_0607 - 복사본.JPG,LotteWorldTower,0.988295


In [ ]:
# Image (5).jpg	
# HF030004_0301_0551 - 복사본.JPG	
# KakaoTalk_20250909_173747263
# KakaoTalk_20250909_173747263_02.jpg	
# HF030004_0201_0190.JPG 근데 이건 사진이 좀..
# KakaoTalk_20250909_173747263_03.jpg	
# KakaoTalk_20250909_173747263_01.jpg	
# KakaoTalk_20250909_173747263_04.jpg	
# HF030004_0301_0123 - 복사본.JPG	
# HF030004_0301_0118 - 복사본.JPG	
# HF030004_0301_0105 - 복사본.JPG	
# HF030004_0301_0110 - 복사본.JPG	
# HF030004_0301_0116 - 복사본.JPG	
# HF030004_0301_0008 - 복사본.JPG	
# HF030001_0101_0201.jpg	
# HF030004_0301_0007 - 복사본.JPG	

In [25]:
low_conf_df = df[df["confidence"] < 0.8]
low_conf_df


,filename,label,confidence
6,다운로드 (23).jpg,GyeongbokgungPalace,0.577644
10,Image (5).jpg,NSeoulTower,0.619295
17,HF030004_0201_0278.JPG,LotteWorldTower,0.771094
22,HF030004_0301_0551 - 복사본.JPG,63Building,0.467161
32,HF030004_0301_0695 - 복사본.JPG,LotteWorldTower,0.781685
39,KakaoTalk_20250909_173747263.jpg,NSeoulTower,0.318858
60,KakaoTalk_20250909_173747263_02.jpg,NSeoulTower,0.444054
61,HF030003_0601_0329.JPG,NSeoulTower,0.781155
68,HF030004_0201_0190.JPG,63Building,0.745828
78,KakaoTalk_20250909_173747263_03.jpg,NSeoulTower,0.495568
